# Conexão com BigQuery
* Base de dados usada: [Resultado da Arrecadação Federal
](https://basedosdados.org/dataset/ab4af450-6b41-412e-b7cb-ec7030646c3d?table=17405008-7635-4b38-a14b-bdfcf78351b2)
* Utilizada própria biblioteca basedosdados no python.
* Utilizando ID do BigQuery (`mensal-arrecadacao-receita`) no Google Cloud (em vez de subir o arquivo csv no drive).

* Extração dos dados


In [ ]:
!pip install basedosdados
import basedosdados as bd
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime

ano_inicio = 2016
ano_fim = 2024
mes_limite_fim = 1

project_id = "mensal-arrecadacao-receita" # id do meu googlecloud

query_uf = f"""
SELECT
    dados.ano, dados.mes, dados.sigla_uf, diretorio_sigla_uf.nome AS nome_uf, dados.cide_combustiveis,
    dados.imposto_importacao, dados.ipi_fumo, dados.ipi_bebidas, dados.ipi_automoveis,
    dados.irpf, dados.irpj_demais_empresas, dados.cofins, dados.pis_pasep,
    diretorio_sigla_uf.regiao AS regiao
FROM `basedosdados.br_rf_arrecadacao.uf` AS dados
LEFT JOIN (SELECT DISTINCT sigla, nome, regiao FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
WHERE dados.ano >= {ano_inicio} AND dados.ano <= {ano_fim}
"""

df_raw = bd.read_sql(query_uf, billing_project_id=project_id, reauth=True)

stg_metadata = {
    "origem": "Base dos Dados - br_rf_arrecadacao.uf",
    "data_coleta": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "versao_api": "v2"
}
print(f"Dados extraídos com sucesso. Metadados: {stg_metadata}")

Downloading: 100%|██████████|
Dados extraídos com sucesso. Metadados: {'origem': 'Base dos Dados - br_rf_arrecadacao.uf', 'data_coleta': '2026-03-07 19:56:09', 'versao_api': 'v2'}


# Criando as Surrogate Keys e a Dimensão Tempo
* Mapeamos o mês/ano da arrecadação para o 1º dia de cada mês. (Era obrigatório 'dia' na descrição do projeto na blackboard)

In [ ]:
#criando as sks e dimensao de tempo no nivel dia
def generate_dim_tempo(start_date, end_date):
    dates = pd.date_range(start=start_date, end=end_date, freq='D')
    df = pd.DataFrame({'data': dates})
    df['SK_Tempo'] = df['data'].dt.strftime('%Y%m%d').astype(int)
    df['BK_Data'] = df['data']
    df['ano'] = df['data'].dt.year
    df['mes'] = df['data'].dt.month
    df['dia'] = df['data'].dt.day
    df['trimestre'] = df['data'].dt.quarter
    df['nome_mes'] = df['data'].dt.month_name()
    return df

#nivel dia: mapearemos o mes/ano da arrecadação para o primeiro dia de cada mês
dim_tempo = generate_dim_tempo('2016-01-01', '2025-12-31')

## Dimensão Localidade / Controlde de Histórico SCD 2

In [ ]:
#criando a dimensao localidade/controle de historico scd tipo2
dim_localidade = df_raw[['sigla_uf', 'nome_uf', 'regiao']].drop_duplicates().reset_index(drop=True)
#dim_localidade = df_raw[['sigla_uf', 'nome_uf']].drop_duplicates().reset_index(drop=True)
dim_localidade['SK_Localidade'] = range(1, len(dim_localidade) + 1)
dim_localidade['data_inicio'] = datetime(2016, 1, 1)
dim_localidade['data_fim'] = datetime(9999, 12, 31)
dim_localidade['is_current'] = 1

## Tabela Fato (Unpivot)
* Para o star scheme foi necessário transformas as colunas de impostos em uma única métrica.


In [ ]:
#p star scheme transformar as colunas de impostos em uma única métrica
cols_tributos = ['imposto_importacao', 'ipi_fumo', 'ipi_bebidas', 'ipi_automoveis', 'cide_combustiveis', 'irpf', 'irpj_demais_empresas', 'cofins', 'pis_pasep']
cols_id = ['ano', 'mes', 'sigla_uf']

df_long = pd.melt(df_raw, id_vars=cols_id, value_vars=cols_tributos, var_name='BK_Tributo', value_name='valor')

dim_tributo = pd.DataFrame({'BK_Tributo': cols_tributos})
dim_tributo['SK_Tributo'] = range(1, len(dim_tributo) + 1)
dim_tributo['descricao'] = dim_tributo['BK_Tributo'].str.replace('_', ' ').str.upper()

fato_arrecadacao = df_long.merge(dim_localidade, on='sigla_uf') \
                          .merge(dim_tributo, on='BK_Tributo')

fato_arrecadacao['data_fake'] = pd.to_datetime(fato_arrecadacao['ano'].astype(str) + '-' + fato_arrecadacao['mes'].astype(str) + '-01')
fato_arrecadacao['SK_Tempo'] = fato_arrecadacao['data_fake'].dt.strftime('%Y%m%d').astype(int)

fato_arrecadacao = fato_arrecadacao[['SK_Tempo', 'SK_Localidade', 'SK_Tributo', 'valor']]

# Carga SGDB e Análise OLAP

*   `Slice`: Filtra apenas a arrecadação para sigla_uf = 'SP'
*   `Dice`: Seleciona imposto_importacao e imposto_exportacao para as UFs RJ, SP e MG no ano de 2023.
* `Drill-down`: Analisa o total anual e detalha por mês.
* `Roll-up`: Agrupa os valores das UFs para visualizar o total por região.


---
**SQLite dentro do Colab para garantir a reprodutividade das consultas analíticas.**






In [ ]:
# para db: sqlite dentro do colab p garantir a reprodutibilidade das consultas analíticas (projeto)
conn = sqlite3.connect('dw_arrecadacao.db')

dim_tempo.to_sql('dim_tempo', conn, if_exists='replace', index=False)
dim_localidade.to_sql('dim_localidade', conn, if_exists='replace', index=False)
dim_tributo.to_sql('dim_tributo', conn, if_exists='replace', index=False)
fato_arrecadacao.to_sql('fato_arrecadacao', conn, if_exists='replace', index=False)

def run_olap(query, title):
    print(f"\n────୨ৎ──── Operação OLAP: {title} ────୨ৎ────")
    display(pd.read_sql(query, conn))

# SLICE: fatia apenas arrecadacoes de sp
query_slice = """
SELECT t.nome_mes, SUM(f.valor) as total
FROM fato_arrecadacao f
JOIN dim_localidade l ON f.SK_Localidade = l.SK_Localidade
JOIN dim_tempo t ON f.SK_Tempo = t.SK_Tempo
WHERE l.sigla_uf = 'SP' AND t.ano = 2023
GROUP BY t.mes
"""
run_olap(query_slice, "SLICE (SP em 2023)")

# DICE: dado:sp e rj, irpf e ipi fumo
query_dice = """
SELECT l.sigla_uf, tr.descricao, SUM(f.valor) as total
FROM fato_arrecadacao f
JOIN dim_localidade l ON f.SK_Localidade = l.SK_Localidade
JOIN dim_tributo tr ON f.SK_Tributo = tr.SK_Tributo
WHERE l.sigla_uf IN ('SP', 'RJ') AND tr.BK_Tributo IN ('irpf', 'ipi_fumo')
GROUP BY l.sigla_uf, tr.descricao
"""
run_olap(query_dice, "DICE (SP/RJ + IRPF/IPI)")


# ROLL-UP: agregação: total por ano
query_rollup = """
SELECT t.ano, SUM(f.valor) as total_nacional
FROM fato_arrecadacao f
JOIN dim_tempo t ON f.SK_Tempo = t.SK_Tempo
GROUP BY t.ano
"""
run_olap(query_rollup, "ROLL-UP (Total Anual Nacional)")

# DRILL-DOWN: detalhamnto: ano para mes
query_drill = """
SELECT t.ano, t.mes, SUM(f.valor) as total
FROM fato_arrecadacao f
JOIN dim_tempo t ON f.SK_Tempo = t.SK_Tempo
WHERE t.ano = 2023
GROUP BY t.ano, t.mes
"""
run_olap(query_drill, "DRILL-DOWN (Mensal dentro de 2023)")


────୨ৎ──── Operação OLAP: SLICE (SP em 2023) ────୨ৎ────


,nome_mes,total
0,January,1.954717e+10
1,February,9.209982e+09
2,March,9.575845e+09
3,April,1.535705e+10
4,May,1.272047e+10
5,June,9.242628e+09
6,July,1.583557e+10
7,August,9.793505e+09
8,September,9.688298e+09
9,October,1.686921e+10



────୨ৎ──── Operação OLAP: DICE (SP/RJ + IRPF/IPI) ────୨ৎ────


,sigla_uf,descricao,total
0,RJ,IPI FUMO,9.100887e+07
1,RJ,IRPF,4.662695e+10
2,SP,IPI FUMO,9.407455e+07
3,SP,IRPF,1.449499e+11



────୨ৎ──── Operação OLAP: ROLL-UP (Total Anual Nacional) ────୨ৎ────


,ano,total_nacional
0,2016,1.954384e+11
1,2017,1.832543e+11
2,2018,2.142523e+11
3,2019,2.410418e+11
4,2020,2.525367e+11
5,2021,3.582745e+11
6,2022,4.208835e+11
7,2023,3.973507e+11
8,2024,2.022316e+11



────୨ৎ──── Operação OLAP: DRILL-DOWN (Mensal dentro de 2023) ────୨ৎ────


,ano,mes,total
0,2023,1,6.034032e+10
1,2023,2,2.497577e+10
2,2023,3,2.577020e+10
3,2023,4,4.009715e+10
4,2023,5,3.533506e+10
5,2023,6,2.500263e+10
6,2023,7,3.942079e+10
7,2023,8,2.769592e+10
8,2023,9,2.485667e+10
9,2023,10,4.334771e+10


# Saneamento de Dados
* Formantando a tabela, tratamento de nulos [...]

In [ ]:
#limpeza
# formatndo a tabela fato p exportacao
fato_final = fato_arrecadacao.copy()

# problema identificado: tratamento de nulos
fato_final['valor'] = fato_final['valor'].fillna(0)

# add metadados de auditoria
fato_final['dt_carga'] = datetime.now()
fato_final['origem_extração'] = "Base dos Dados - RFB"

# verificação de integridade
# garante q não tem SK_Tempo nula que quebraria o relacionamento no bi
assert fato_final['SK_Tempo'].isnull().sum() == 0, "Erro: Há chaves de tempo nulas!"

print("Dados saneados e prontos para carga final.")

Dados saneados e prontos para carga final.


# Teste: Google Cloud

In [ ]:
#preparando ambiente p googlecloud
from google.cloud import bigquery
from google.api_core.exceptions import Conflict
import pandas as pd

def initialize_dw_dataset(project_id, dataset_name):
    client = bigquery.Client(project=project_id)
    dataset_id = f"{project_id}.{dataset_name}"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = "US"

    try:
        dataset = client.create_dataset(dataset, timeout=30)
    except Conflict:
        dataset = client.get_dataset(dataset_id)

    return dataset_id

def upload_warehouse_to_bq(project_id, dataset_name, tables_dict):
    for table_name, dataframe in tables_dict.items():
        destination = f"{dataset_name}.{table_name}"
        dataframe.to_gbq(
            destination_table=destination,
            project_id=project_id,
            if_exists='replace',
            progress_bar=True
        )

# execucao
my_project = "mensal-arrecadacao-receita" #igbigquery meu googlecloud
my_dataset = "dw_arrecadacao_rfb"

tables_to_upload = {
    "dim_tempo": dim_tempo,
    "dim_localidade": dim_localidade,
    "dim_tributo": dim_tributo,
    "fato_arrecadacao": fato_final
}

ds_id = initialize_dw_dataset(my_project, my_dataset)
upload_warehouse_to_bq(my_project, my_dataset, tables_to_upload)

100%|██████████| 1/1 [00:00<00:00, 9686.61it/s]


# União de Dados
fato, dimensões e merge dw completo

In [ ]:
# unido a fato com a dimensão de tempo
df_final = fato_arrecadacao.merge(dim_tempo, on='SK_Tempo')

# verificando se as colunas 'mes' e 'ano' agora existem
print(df_final.columns)

Index(['SK_Tempo', 'SK_Localidade', 'SK_Tributo', 'valor', 'data', 'BK_Data',
       'ano', 'mes', 'dia', 'trimestre', 'nome_mes'],
      dtype='object')


In [ ]:
# unindo com a dimensão de localidade para pegar a sigla_uf (grafico ranking estados)
df_final = df_final.merge(dim_localidade, on='SK_Localidade')

print(df_final.columns)

Index(['SK_Tempo', 'SK_Localidade', 'SK_Tributo', 'valor', 'data', 'BK_Data',
       'ano', 'mes', 'dia', 'trimestre', 'nome_mes', 'sigla_uf', 'nome_uf',
       'regiao', 'data_inicio', 'data_fim', 'is_current'],
      dtype='object')


In [ ]:
# criando o 'DW_Completo'
df_completo = fato_arrecadacao.merge(dim_tempo, on='SK_Tempo') \
                             .merge(dim_localidade, on='SK_Localidade') \
                             .merge(dim_tributo, on='SK_Tributo')

# Cálculo de Indicadores (KPIs)
* **Arrecadação Total**: Volume absoluto de recursos captados.
* **Market Share por UF**: Relevância do estado no total federal.
* **Mix de Impostos**: Composição da carga tributária por categoria.
* **Arrecadação per Capita**: Eficiência de Arrecadação por habitante (cruzamento)
* **Crescimento YoY**: Variação anual da arrecadação.

## KPIS (1, 2, 3 e 4)

In [ ]:

# arrecadação total
total_nacional = df_final['valor'].sum()
print(f"KPI 1 - Arrecadação Total: R$ {total_nacional/1e12:.2f} Trilhões")

#crescimento yoy 2023vs20222 (pq 2024 ta incompleto)
valor_2022 = df_final[df_final['ano'] == 2022]['valor'].sum()
valor_2023 = df_final[df_final['ano'] == 2023]['valor'].sum()
crescimento_yoy = ((valor_2023 / valor_2022) - 1) * 100
print(f"KPI 2 - Crescimento YoY: {crescimento_yoy:.2f}%")

#market share por uf(%)
df_market_share = (df_final.groupby('sigla_uf')['valor'].sum() / total_nacional) * 100
print("\nKPI 3 - Market Share (Top 5 UFs):")
print(df_market_share.sort_values(ascending=False).head(5))

print("\nKPI SAD:")
print(df_final.columns)

# Use o df_completo da "União de Dados"
df_mix_impostos = (df_completo.groupby('descricao')['valor'].sum() / total_nacional) * 100

print("\nKPI 4 - Mix de Impostos (Principais):")
print(df_mix_impostos.sort_values(ascending=False).head(5))


KPI 1 - Arrecadação Total: R$ 2.47 Trilhões
KPI 2 - Crescimento YoY: -5.59%

KPI 3 - Market Share (Top 5 UFs):
sigla_uf
SP    38.230258
RJ    17.645589
MG     8.537212
SC     6.216570
PR     5.729039
Name: valor, dtype: float64

KPI SAD:
Index(['SK_Tempo', 'SK_Localidade', 'SK_Tributo', 'valor', 'data', 'BK_Data',
       'ano', 'mes', 'dia', 'trimestre', 'nome_mes', 'sigla_uf', 'nome_uf',
       'regiao', 'data_inicio', 'data_fim', 'is_current'],
      dtype='object')

KPI 4 - Mix de Impostos (Principais):
descricao
IRPJ DEMAIS EMPRESAS    63.356832
IMPOSTO IMPORTACAO      16.060914
IRPF                    15.567593
IPI FUMO                 1.852592
IPI AUTOMOVEIS           1.232004
Name: valor, dtype: float64


## KPI 5 ❓

In [ ]:

## NAO ESTA FUNCIONANDO @@@@@@@@@@@@@@@@@@@@@@@@
#arrecadacao per capita - cruzqamento c populacao d dim_localidade
# puxando dados do IBGE direto do BigQuery
query_pop = """
SELECT ano, sigla_uf, populacao
FROM `basedosdados.br_ibge_populacao.uf`
WHERE ano BETWEEN 2016 AND 2024
"""
df_populacao = pd.read_gbq(query_pop, project_id='mensal-arrecadacao-receita')

# unindo dataframe real
# merge por Ano e UF para garantir que o cálculo per capita seja correto
df_completo_with_pop = df_completo.merge(df_populacao, on=['ano', 'sigla_uf'], how='left')

# KPI 5: Arrecadação per Capita
##TODO
# formula: Soma(Valor_UF) / Populacao_UF
arrec_por_uf = df_completo_with_pop.groupby(['sigla_uf', 'ano'])[['valor', 'populacao']].agg({'valor': 'sum', 'populacao': 'first'}).reset_index()
arrec_por_uf['per_capita'] = arrec_por_uf['valor'] / arrec_por_uf['populacao']

print("KPI 5 - Arrecadação per Capita (Top 5 UFs - R$/Habitantes):")
print(arrec_por_uf.sort_values(by='per_capita', ascending=False).head(5))


KPI 5 - Arrecadação per Capita (Top 5 UFs - R$/Habitantes):
    sigla_uf   ano         valor  populacao   per_capita
54        DF  2016  2.881722e+10    2977216  9679.250347
168       RJ  2022  9.508534e+10   16055174  5922.410803
169       RJ  2023  8.005305e+10   16055174  4986.121541
212       SC  2021  2.510473e+10    7338473  3420.974815
214       SC  2023  2.579210e+10    7610361  3389.076693


### Teste (IGNORAR POR AGORA)

In [ ]:
#--------------------------WARNING--------------------------
# Testa Dps isso !
#--------------------------WARNING--------------------------

# ano_ref = arrec_por_uf['ano'].max()

# kpi5_uf = (
#     arrec_por_uf[arrec_por_uf['ano'] == ano_ref]
#     .sort_values('per_capita', ascending=False)
#     .reset_index(drop=True)
# )
# kpi5_uf.index += 1  # ranking começa em 1

# print(f"KPI 5 - Arrecadação per Capita por UF ({ano_ref})")
# display(
#     kpi5_uf[['sigla_uf', 'populacao' ,'per_capita']]
#     .rename(columns={
#         'sigla_uf': 'UF',
#         'populacao': 'População',
#         'per_capita': 'R$/Habitante'})
#     .style.format({'População': '{:.0f}',
#                     'R$/Habitante': 'R$ {:,.2f}'
#                    'População': '{:.0f}'})
# )

# Gráfico Sazonalidade da Arrecadação
Flutuações previsíveis na demanda por produtos ou serviços, com picos ou quedas significativas de vendas em períodos específicos do ano influenciados pelo clima, feriados ou hábitos culturais.

In [ ]:
import plotly.express as px

df_final = fato_arrecadacao.merge(dim_tempo, on='SK_Tempo')
df_sazonalidade = df_final.groupby(['ano', 'mes'])['valor'].sum().reset_index()

# nova coluna c valor bilhoes
df_sazonalidade['valor_B'] = df_sazonalidade['valor'] / 1e9

# grafico usa essa nova coluna
fig = px.line(df_sazonalidade,
              x="mes",
              y="valor_B", # coluna escalada
              color="ano",
              title="Sazonalidade da Arrecadação (em Bilhões R$)",
              markers=True)

# formatando p mostrar B de bilhoes
fig.update_layout(
    yaxis_tickformat='.2f',  # 2casas
    yaxis_ticksuffix=" B",
    yaxis_title="Bilhões de Reais (R$)"
)

fig.update_xaxes(tickmode='linear', tick0=1, dtick=1)
fig.show()

# Gráfico Top 5 Estados com maior Arrecadaçao Acumulada

In [ ]:
import plotly.express as px

# agrupando por estados e somando valores
df_ranking = df_completo.groupby('sigla_uf')['valor'].sum().reset_index()

# converte p bilhoes (p contornar o problema de aparecr gigas em vez d bilhoes)
df_ranking['valor_B'] = df_ranking['valor'] / 1e9
df_top5 = df_ranking.sort_values(by='valor_B', ascending=False).head(5)

# ráfico de barras
fig_ranking = px.bar(
    df_top5,
    x='sigla_uf',
    y='valor_B',
    title="Top 5 Estados com Maior Arrecadação (Acumulado)",
    labels={'valor_B': 'Total em Bilhões (R$)', 'sigla_uf': 'Estado'},
    color='valor_B',
    text_auto='.2f',
    color_continuous_scale='Blues'  # paleta
)

fig_ranking.update_layout(yaxis_ticksuffix=" B")
fig_ranking.show()

# Gráfico Participação por Tipo de Tributo em Bilhões


In [ ]:
import plotly.express as px

df_tributo_rank = df_completo.groupby('descricao')['valor'].sum().reset_index()
df_tributo_rank['valor_B'] = df_tributo_rank['valor'] / 1e9
df_tributo_rank = df_tributo_rank.sort_values(by='valor_B', ascending=True)

fig_tributos = px.bar(df_tributo_rank,
                      x='valor_B',
                      y='descricao',
                      orientation='h',
                      title="Participação por Tipo de Tributo (em Bilhões de R$)",
                      labels={'valor_B': 'Total (R$ Bilhões)', 'descricao': 'Tributo'},
                      text_auto='.2f',
                      color='valor_B',
                      color_continuous_scale='Blues')

fig_tributos.update_layout(yaxis={'categoryorder':'total ascending'}, height=600)
fig_tributos.show()

# Arrecadação Total

In [ ]:
# deixar isso por ultimo

total_geral = df_final['valor'].sum() / 1e12 # divide por 1 trilhão
print(f"Arrecadação Total Acumulada: R$ {total_geral:.2f} Trilhões")
#resumo da arrecadação total

Arrecadação Total Acumulada: R$ 2.47 Trilhões


# Perguntas Analíticas Levantadas

## Evolução Mensal da Arrecadação por UF ❓
**Questão 1:** Qual a evolução mensal da arrecadação federal total por Unidade da Federação (UF)?

In [ ]:
#resposta
df_evol_uf = df_completo.groupby(['ano', 'mes', 'sigla_uf'])['valor'].sum().reset_index()
df_evol_uf['valor_B'] = df_evol_uf['valor'] / 1e9

fig1 = px.line(df_evol_uf, x="mes", y="valor_B", color="sigla_uf",
              facet_col="ano", title="Evolução Mensal da Arrecadação por UF (Bilhões R$)")
fig1.update_layout(yaxis_ticksuffix=" B")
fig1.show()

## Crescimento 2022 vs. 2023
**Questão 2**: Quais estados apresentaram o maior crescimento percentual de arrecadação comparando 2023 vs. 2022?

In [ ]:
# resposta
df_jan_mai = df_completo[df_completo['mes'] <= 12] #1 ano
growth_uf = df_jan_mai.pivot_table(index='sigla_uf', columns='ano', values='valor', aggfunc='sum')
growth_uf['pct_growth'] = ((growth_uf[2023] / growth_uf[2022]) - 1) * 100
print(growth_uf['pct_growth'].sort_values(ascending=False).head(5))

sigla_uf
RO    63.874037
AL    49.725049
PB    19.201075
PI    14.924066
RN    12.270723
Name: pct_growth, dtype: float64


## Participação do Imposto de Importação por Região: DANDO ERRO ❓
**Questão 3:** Qual a participação percentual do Imposto de Importação na arrecadação total de cada região?
* KPI: Market Share

In [ ]:
#% do imposto de importacao na região filtrado pelo nome do tributo na dim_tributo
df_ii = df_completo[df_completo['descricao'].str.contains('IMPORTACAO', na=False)]

total_regiao = df_completo.groupby('regiao')['valor'].sum()
ii_regiao = df_ii.groupby('regiao')['valor'].sum()

percentual_ii = (ii_regiao / total_regiao) * 100
print("Participação do Imposto de Importação por Região (%):\n", percentual_ii.sort_values(ascending=False))

Participação do Imposto de Importação por Região (%):
 regiao
Sul             28.187737
Nordeste        15.187859
Sudeste         14.420856
Norte           13.466963
Centro-Oeste     3.374871
Name: valor, dtype: float64


## Análise Setorial de IPI: Autos, Bebidas e Fumo
**Questão 4**: Existe sazonalidade identificável na arrecadação de IPI (Automóveis e Bebidas) ao longo dos anos?

**Questão 5**: Como a arrecadação de IPI-Fumo se comporta em relação aos outros setores industriais tributados?

In [ ]:
#Sazonalidade IPI
# lista de setores p filtrar
setores_ipi = ['AUTOMOVEIS', 'BEBIDAS', 'FUMO']
df_setores = df_completo[df_completo['descricao'].str.contains('|'.join(setores_ipi), na=False)]

df_ipi_sazonal = df_setores.groupby(['ano', 'mes', 'descricao'])['valor'].sum().reset_index()

fig_ipi = px.line(df_ipi_sazonal, x='mes', y='valor', color='ano', facet_col='descricao',
                  title="Sazonalidade IPI: Fumo vs Automóveis vs Bebidas")
fig_ipi.update_yaxes(matches=None) # escalas independentes para ver melhor as variações
fig_ipi.show()

## UFs responsáveis por >50% da arrecadação
**Questão 6 :** Quais UFs são responsáveis por mais de 50% da arrecadação total do país?

KPI 3 (market share): Pelos dados, São Paulo e Rio de Janeiro dominam o volume total.


In [ ]:
# resposta a questao 6
df_market = df_completo.groupby('sigla_uf')['valor'].sum().sort_values(ascending=False).reset_index()
df_market['share_acumulado'] = (df_market['valor'].cumsum() / df_market['valor'].sum()) * 100

# filta so quem faz parte dos primeiros 50%
top_50_ufs = df_market[df_market['share_acumulado'] <= 55] # Margem para pegar as principais
print(top_50_ufs[['sigla_uf', 'share_acumulado']])

  sigla_uf  share_acumulado
0       SP        38.230258


### Peso do IPI-Bebidas por Estação do Ano
* **Questão 7:** Qual o peso do IPI-Bebidas na arrecadação total nos meses de verão vs inverno no recorte temporal de 2016 a 2024?


In [ ]:

import plotly.graph_objects as go

# map de estações (hemisfério sul)
estacoes_map = {
    12: 'Verão', 1: 'Verão', 2: 'Verão',
    3: 'Outono', 4: 'Outono', 5: 'Outono',
    6: 'Inverno', 7: 'Inverno', 8: 'Inverno',
    9: 'Primavera', 10: 'Primavera', 11: 'Primavera'
}
df_completo['estacao'] = df_completo['mes'].map(estacoes_map)

# total geral por mês/ano
total_mensal = df_completo.groupby(['ano', 'mes', 'estacao'])['valor'].sum().reset_index()
total_mensal.rename(columns={'valor': 'total_geral'}, inplace=True)

# IPI-Bebidas por mês/ano
df_bebidas = df_completo[df_completo['descricao'].str.contains('BEBIDAS', na=False)]
bebidas_mensal = df_bebidas.groupby(['ano', 'mes'])['valor'].sum().reset_index()
bebidas_mensal.rename(columns={'valor': 'valor_bebidas'}, inplace=True)

# merge e cálculo do peso %
df_peso = total_mensal.merge(bebidas_mensal, on=['ano', 'mes'])
df_peso['peso_pct'] = (df_peso['valor_bebidas'] / df_peso['total_geral']) * 100
df_peso['data'] = pd.to_datetime(df_peso['ano'].astype(str) + '-' + df_peso['mes'].astype(str) + '-01')
df_peso = df_peso.sort_values('data')

cor_estacao = {
    'Verão': '#f97316',
    'Outono': '#a3a3a3',
    'Inverno': '#3b82f6',
    'Primavera': '#22c55e'
}
df_peso['cor'] = df_peso['estacao'].map(cor_estacao)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_peso['data'],
    y=df_peso['peso_pct'],
    mode='lines',
    line=dict(color='lightgray', width=1.5),
    showlegend=False,
    hoverinfo='skip'
))

for estacao, cor in cor_estacao.items():
    mask = df_peso['estacao'] == estacao
    fig.add_trace(go.Scatter(
        x=df_peso[mask]['data'],
        y=df_peso[mask]['peso_pct'],
        mode='markers',
        name=estacao,
        marker=dict(color=cor, size=8),
        hovertemplate='<b>%{x|%b/%Y}</b><br>Peso: %{y:.2f}%<extra></extra>'
    ))

# media de verão e inverno como linhas horizontais de referência
media_verao  = df_peso[df_peso['estacao'] == 'Verão']['peso_pct'].mean()
media_inverno = df_peso[df_peso['estacao'] == 'Inverno']['peso_pct'].mean()

fig.add_hline(y=media_verao, line_dash='dot', line_color='#f97316',
              annotation_text=f'Média Verão: {media_verao:.2f}%',
              annotation_position='top right')

fig.add_hline(y=media_inverno, line_dash='dot', line_color='#3b82f6',
              annotation_text=f'Média Inverno: {media_inverno:.2f}%',
              annotation_position='bottom right')

fig.update_layout(
    title='Peso % do IPI-Bebidas na Arrecadação Total de 2016 a 2024 (mês a mês)',
    xaxis_title='Mês/Ano',
    yaxis_title='% da Arrecadação Total',
    xaxis=dict(tickformat='%b/%Y', tickangle=-45),
    height=500,
    hovermode='x unified',
    legend_title='Estação'
)
fig.show()

print(f"\nMédia do peso do IPI-Bebidas:")
print(f"   Verão  (Dez/Jan/Fev): {media_verao:.2f}%")
print(f"   Inverno (Jun/Jul/Ago): {media_inverno:.2f}%")
print(f"   Diferença: {media_verao - media_inverno:.2f} p.p.")


Média do peso do IPI-Bebidas:
   Verão  (Dez/Jan/Fev): 1.31%
   Inverno (Jun/Jul/Ago): 0.93%
   Diferença: 0.38 p.p.


### UF's Dependentes de Um Único Tributo
* **Questão 8:** Quais UF’s são mais dependentes de um único tributo (concentração de receita)?

In [ ]:

import plotly.express as px

# participacao de cada tributo dentro de cada UF
total_por_uf = df_completo.groupby('sigla_uf')['valor'].sum().reset_index()
total_por_uf.rename(columns={'valor': 'total_uf'}, inplace=True)

tributo_por_uf = df_completo.groupby(['sigla_uf', 'descricao'])['valor'].sum().reset_index()
tributo_por_uf = tributo_por_uf.merge(total_por_uf, on='sigla_uf')
tributo_por_uf['share_pct'] = (tributo_por_uf['valor'] / tributo_por_uf['total_uf']) * 100

# p cada UF: pega o tributo dominante (maior share)
dominante = (
    tributo_por_uf.sort_values('share_pct', ascending=False)
    .groupby('sigla_uf')
    .first()
    .reset_index()
    [['sigla_uf', 'descricao', 'share_pct']]
    .rename(columns={'descricao': 'tributo_dominante'})
    .sort_values('share_pct', ascending=True)
)

cores_tributo = {
    'COFINS':                  '#3b82f6',
    'PIS PASEP':               '#8b5cf6',
    'IRPF':                    '#f97316',
    'IRPJ DEMAIS EMPRESAS':    '#22c55e',
    'IMPOSTO IMPORTACAO':      '#ef4444',
    'IPI AUTOMOVEIS':          '#eab308',
    'IPI FUMO':                '#6b7280',
    'IPI BEBIDAS':             '#ec4899',
    'CIDE COMBUSTIVEIS':       '#14b8a6',
}
dominante['cor'] = dominante['tributo_dominante'].map(cores_tributo).fillna('#94a3b8')

fig = go.Figure()

fig.add_trace(go.Bar(
    x=dominante['share_pct'],
    y=dominante['sigla_uf'],
    orientation='h',
    marker_color=dominante['cor'],
    text=dominante['tributo_dominante'] + '  ' + dominante['share_pct'].round(1).astype(str) + '%',
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Tributo dominante: %{text}<extra></extra>'
))

fig.add_vline(
    x=50, line_dash='dash', line_color='red',
    annotation_text='50% — limiar de dependência',
    annotation_position='top'
)

fig.update_layout(
    title='Concentração de Receita por UF — Qual tributo domina cada estado?',
    xaxis_title='Share do tributo dominante na arrecadação total da UF (%)',
    yaxis_title='',
    height=650,
    margin=dict(r=250),
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#e5e7eb', range=[0, 105])
)

fig.show()

print("\nUFs mais dependentes de um único tributo:")
print(dominante[['sigla_uf', 'tributo_dominante', 'share_pct']]
      .sort_values('share_pct', ascending=False)
      .head(5)
      .to_string(index=False))


UFs mais dependentes de um único tributo:
sigla_uf    tributo_dominante  share_pct
      DF IRPJ DEMAIS EMPRESAS  87.985656
      RJ IRPJ DEMAIS EMPRESAS  77.518726
      MT IRPJ DEMAIS EMPRESAS  69.637898
      AM IRPJ DEMAIS EMPRESAS  68.538982
      RR IRPJ DEMAIS EMPRESAS  68.114158


## Sazonalidade Mensal por Ano no Recorte Temporal de 2016-2024
* **Questão 9**: Qual mês do ano historicamente concentra mais arrecadação?

In [ ]:
import plotly.graph_objects as go

# Questão 9 — Composição mensal da arrecadação por ano (2016–2024)
df_mes_ano = df_completo[df_completo['ano'].between(2016, 2024)].groupby(['ano', 'mes'])['valor'].sum().reset_index()
df_mes_ano['valor_B'] = df_mes_ano['valor'] / 1e9
df_mes_ano['nome_mes'] = df_mes_ano['mes'].map({
    1:'Jan', 2:'Fev', 3:'Mar', 4:'Abr', 5:'Mai', 6:'Jun',
    7:'Jul', 8:'Ago', 9:'Set', 10:'Out', 11:'Nov', 12:'Dez'
})

cores = {
    2016: 'steelblue',  2017: 'darkorange', 2018: 'green',
    2019: 'red',        2020: 'purple',     2021: 'brown',
    2022: 'teal',       2023: 'crimson',    2024: 'gold'
}

fig = go.Figure()

for ano in sorted(df_mes_ano['ano'].unique()):
    df_ano = df_mes_ano[df_mes_ano['ano'] == ano]
    fig.add_trace(go.Bar(
        x=df_ano['nome_mes'],
        y=df_ano['valor_B'],
        name=str(ano),
        marker_color=cores.get(ano)
    ))

fig.update_layout(
    barmode='stack',
    title='Evolução Mensal da Arrecadação Federal por Ano (2016–2024) — R$ Bilhões',
    xaxis_title='Mês',
    yaxis_title='R$ Bilhões',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)

fig.show()

## Sazonalidade de IPI e IPI-Fumo
**Questão 4:** Existe sazonalidade identificável na arrecadação de IPI (Automóveis e Bebidas) ao longo dos anos?

**Questão 10**: Qual tributo possui a maior volatilidade mensal? [Intensidade e frequência das oscilações de preço de um ativo financeiro em um determinado período]

In [ ]:
# resposta
volatilidade = df_completo.groupby('descricao')['valor'].std() / df_completo.groupby('descricao')['valor'].mean()
print(volatilidade.sort_values(ascending=False).head(5))

descricao
CIDE COMBUSTIVEIS       4.872316
IPI FUMO                4.270121
IPI AUTOMOVEIS          3.231955
IRPJ DEMAIS EMPRESAS    2.742692
IRPF                    2.713780
Name: valor, dtype: float64


# Testes - Combo das Respostas Analíticas